In [7]:
import uproot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle
from scipy.optimize import curve_fit
from tqdm import tqdm

In [5]:
def read_root_file(
        fn,
        treename='events', 
        istart=0, 
        istop=1000
        ):

    with uproot.open(fn) as f:
        tree = f[f"{treename}"][f"{treename}"]
        branches = ['nsteps', 
                    'xp', 
                    'yp', 
                    'zp', 
                    'etot', 
                    'xp_pri', 
                    'yp_pri', 
                    'zp_pri', 
                    'ed',
                    'PreStepEnergy',
                    'type',
                    'time']
        
        data = tree.arrays(
            branches, 
            filter_name='nsteps', 
            library='np', 
            entry_start=istart, 
            entry_stop=istop
            )

    return data

In [118]:
# Parquet filepaths
filepath  = "/home/hargy/Scientific/Projects/Hermetic-TPC/loc_data/raw/Cryostat_U238_0005.root"
data = read_root_file(filepath)
df = pd.DataFrame(data)

In [32]:
keys = data.keys()
keys

dict_keys(['nsteps', 'xp', 'yp', 'zp', 'etot', 'xp_pri', 'yp_pri', 'zp_pri', 'ed', 'PreStepEnergy', 'type', 'time'])

In [47]:
"""
nsteps_arr = []
xp_arr = []
yp_arr = []
zp_arr = []
etot_arr = []
xp_pri_arr = []
yp_pri_arr = []
zp_pri_arr = []
ed_arr = []
PreStepEnergy_arr = []
type_arr = []
time_arr = []
"""

'\nnsteps_arr = []\nxp_arr = []\nyp_arr = []\nzp_arr = []\netot_arr = []\nxp_pri_arr = []\nyp_pri_arr = []\nzp_pri_arr = []\ned_arr = []\nPreStepEnergy_arr = []\ntype_arr = []\ntime_arr = []\n'

In [134]:
def time_cluster(time_arr, scale=1e8):
    sz = len(time_arr)
    order = np.argsort(time_arr)
    times = time_arr[order]

    cluster_num = 0
    cluster_idxs_sorted = np.zeros(sz, dtype=int)

    prev_t = times[0]
    cluster_idxs_sorted[0] = 0

    for i in range(1, sz):
        t = times[i]
        delta = t - prev_t

        if delta > scale:
            cluster_num += 1
        cluster_idxs_sorted[i] = cluster_num
        prev_t = t

    # map cluster labels back to original order
    cluster_idxs = np.zeros(sz, dtype=int)
    cluster_idxs[order] = cluster_idxs_sorted

    return cluster_idxs



def split_by_septime(data, scale=1e8):

    branches = ['nsteps', 
                'xp', 
                'yp', 
                'zp', 
                'etot', 
                'xp_pri', 
                'yp_pri', 
                'zp_pri', 
                'ed',
                'PreStepEnergy',
                'type',
                'time']

    val_mtx = [[] for _ in range(12)]

    time_mtx = np.array(data['time'])

    for i, time in enumerate(time_mtx):
        cluster_idxs = time_cluster(time, scale=scale)
        t_idx_uniques = np.unique(cluster_idxs)

        for t_idx in t_idx_uniques:
            mask = (cluster_idxs == t_idx)
            for k in range(len(val_mtx)):
                old_val = data[branches[k]][i]
                if isinstance(old_val, np.ndarray):
                    new_val = old_val[mask]
                elif isinstance(old_val, (np.ndarray, list)):
                    new_val = old_val[mask]
                else:
                    new_val = old_val

                val_mtx[k].append(new_val)

    new_data = {
        branches[0]  : val_mtx[0],
        branches[1]  : val_mtx[1],
        branches[2]  : val_mtx[2],
        branches[3]  : val_mtx[3],
        branches[4]  : val_mtx[4],
        branches[5]  : val_mtx[5],
        branches[6]  : val_mtx[6],
        branches[7]  : val_mtx[7],
        branches[8]  : val_mtx[8],
        branches[9]  : val_mtx[9],
        branches[10] : val_mtx[10],
        branches[11] : val_mtx[11]
    }

    return new_data

In [137]:
for i in range(1, 10):
    filepath  = f"/home/hargy/Scientific/Projects/Hermetic-TPC/loc_data/raw/Cryostat_Th232_000{i}.root"
    data = read_root_file(filepath)
    data_septimes = split_by_septime(data)
    df_septimes = pd.DataFrame(data_septimes)
    times = np.array(df_septimes.loc[:, 'time'])

    deltas = []
    for time in times:
        delta = np.max(time) - np.min(time)
        deltas.append(delta)

    deltas = np.array(deltas)

    print(deltas[deltas>0])


[]
[]
[]
[]
[]
[]
[]
[]
[]


In [138]:
df_septimes

,nsteps,xp,yp,zp,etot,xp_pri,yp_pri,zp_pri,ed,PreStepEnergy,type,time
0,1,[-755.80646],[-1004.585],[1365.0],0.000000,-1533.545654,-732.185669,-1614.569458,[0.0],[1042.475],(anti_nu_e),[3.3650093e+17]
1,2,"[-426.10748, -1474.2861]","[1435.0754, -259.78714]","[325.97324, -535.903]",0.000000,1169.995483,1015.864258,-141.715698,"[0.0, 0.0]","[1086.0417, 2159.1682]","(anti_nu_e, anti_nu_e)","[2.7899874e+17, 2.7899874e+17]"
2,1,[1405.1376],[-516.3308],[-217.41302],0.000000,-1462.352905,-496.603973,1654.391968,[0.0],[1967.4861],(anti_nu_e),[2.007921e+17]
3,2,"[-874.01184, 229.20847]","[-1215.3651, 1479.3486]","[555.5821, -1224.1665]",0.000000,1542.306396,57.583878,1350.360107,"[0.0, 0.0]","[269.8889, 1659.0671]","(anti_nu_e, anti_nu_e)","[6712612000000000.0, 6712612000000000.0]"
4,1063,"[-519.34357, -1128.9874, -1125.5963, -1125.596...","[-1404.0269, -955.63654, -963.89777, -963.8977...","[123.762794, 492.56116, 478.2707, 478.2707, 47...",911.209167,-1216.184814,-948.607910,514.130615,"[0.0, 0.0, 0.22059, 0.02056, 0.02056, 0.02056,...","[902.15265, 911.20905, 454.31168, 0.02056, 0.0...","(anti_nu_e, gamma, gamma, e-, e-, e-, e-, e-, ...","[2.1030488e+18, 2.1030488e+18, 2.1030488e+18, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...
1000,1,[-1420.624],[472.05536],[-507.3316],0.000000,1121.411499,-1379.295532,1831.530640,[0.0],[1063.1759],(anti_nu_e),[1.843704e+17]
1001,4,"[1111.7765, -188.21161, -141.93027, 337.30045]","[1002.478, 324.34763, 343.86014, 545.90686]","[136.72147, 693.0219, 752.2024, 1365.0]",0.000000,-1522.477173,-238.187149,-1013.118225,"[0.0, 0.0, 0.0, 0.0]","[1073.7559, 250.76927, 250.76927, 250.76927]","(anti_nu_e, anti_nu_e, anti_nu_e, anti_nu_e)","[2.975076e+17, 2.975076e+17, 2.975076e+17, 2.9..."
1002,4,"[-402.00592, -251.19083, 368.87024, 1454.3062]","[334.45026, 278.4388, 48.153385, -354.96832]","[-1680.5, -1606.6106, -1302.8219, -771.03046]",0.000000,-1370.070557,693.981262,-2154.787354,"[0.0, 0.0, 0.0, 0.0]","[291.54565, 291.54565, 291.54565, 291.54565]","(anti_nu_e, anti_nu_e, anti_nu_e, anti_nu_e)","[1.6968404e+18, 1.6968404e+18, 1.6968404e+18, ..."
1003,597,"[-462.99844, 141.88718, -1096.1295, -1097.1276...","[1423.6016, 1490.2607, -1016.80365, -1016.3169...","[601.36255, 324.54922, -536.56287, -542.129, -...",480.111664,-1127.551880,-1057.848755,-514.396912,"[0.0, 0.0, 0.0, 0.64887, 0.04586, 0.02056, 0.0...","[12.480431, 894.0865, 480.11194, 240.88112, 0....","(anti_nu_e, anti_nu_e, gamma, gamma, e-, e-, e...","[1.3974073e+17, 1.3974073e+17, 1.3974073e+17, ..."


In [117]:
df_septimes = pd.DataFrame(data_septimes)
times = np.array(df_septimes.loc[:, 'time'])

deltas = []
for time in times:
    delta = np.max(time) - np.min(time)
    deltas.append(delta)

deltas = np.array(deltas)

print(deltas[deltas>0])


[]


In [79]:
df_septimes

,nsteps,xp,yp,zp,etot,xp_pri,yp_pri,zp_pri,ed,PreStepEnergy,type,time
0,1,[-1316.8759],[711.9318],[797.14],0.000000,-400.271362,1523.301025,1735.899170,[0.0],[1188.3193],(anti_nu_e),[1.4857022e+17]
1,4,"[-42.23834, -6.4895716, 18.096273, 18.377333]","[372.61365, -371.9434, -884.00476, -889.8585]","[-711.4914, -1283.8596, -1677.5, -1682.0]",0.000000,-98.337860,1541.025757,186.709747,"[0.0, 0.0, 0.0, 0.0]","[1431.8018, 1431.8018, 1431.8018, 1431.8018]","(anti_nu_e, anti_nu_e, anti_nu_e, anti_nu_e)","[6.0635496e+17, 6.0635496e+17, 6.0635496e+17, ..."
2,4,[-460.05548],[1424.5553],[736.92413],0.000000,-1425.054810,-591.001038,772.520081,[0.0],[166.98573],"(anti_nu_e, anti_nu_e, anti_nu_e, anti_nu_e)",[1.6207593e+18]
3,4,"[-298.26498, -296.1954, -67.080925]","[-1192.514, -1193.6189, -1495.4963]","[-1677.5, -1682.0, 651.2991]",0.000000,-1425.054810,-591.001038,772.520081,"[0.0, 0.0, 0.0]","[936.2934, 936.2934, 3158.7446]","(anti_nu_e, anti_nu_e, anti_nu_e, anti_nu_e)","[1.6207851e+18, 1.6207851e+18, 1.6207851e+18]"
4,592,"[1116.2281, 1113.532, 1107.9775, 1107.1559, 11...","[-830.49646, -824.14185, -813.87646, -811.8868...","[1360.144, 1344.7518, 1341.3301, 1343.4852, 13...",441.043549,1063.262329,-401.349304,2259.567383,"[0.0, 0.0, 0.0, 0.0, 0.30467, 0.02056, 0.02056...","[441.04346, 247.64366, 211.52078, 173.786, 142...","(gamma, gamma, gamma, gamma, gamma, e-, e-, e-...","[1.3474237e+16, 1.3474237e+16, 1.3474237e+16, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...
1423,1,[-797.898],[-1266.6364],[-855.6477],0.000000,-791.689270,1565.261353,7.229473,[0.0],[612.1129],(anti_nu_e),[8.287454e+16]
1424,135,"[-173.23433, 504.19977, 504.19977, 504.19977, ...","[-382.04523, -1408.5593, -1408.5593, -1408.559...","[1365.0, -188.4035, -188.4035, -188.4035, -188...",77.207809,854.696960,-1462.641602,-222.145218,"[0.0, 0.20734, 0.04586, 0.02262, 0.03321, 0.04...","[673.3428, 77.20782, 0.04586, 0.02262, 0.03321...","(anti_nu_e, gamma, e-, e-, e-, e-, e-, e-, e-,...","[4.275673e+17, 4.275673e+17, 4.275673e+17, 4.2..."
1425,1,[1496.1626],[50.064358],[1297.7334],0.000000,-666.407593,-605.735596,2380.067871,[0.0],[175.0249],(anti_nu_e),[2.637364e+16]
1426,194,[1367.1604],[-609.8209],[1251.2682],149.680450,365.063049,-1515.254517,1764.342651,[0.0],[2186.115],"(anti_nu_e, gamma, e-, e-, e-, e-, e-, e-, e-,...",[1.4698299e+17]


In [104]:
time_arr = np.array([1,2,3,4,6,5,7,8,9])
order = np.argsort(time_arr)
time_sorted = time_arr[order]

print(order)
print(time_sorted)

[0 1 2 3 5 4 6 7 8]
[1 2 3 4 5 6 7 8 9]
